In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, classification_report

X= pd.read_csv("../data/features/X.csv")
y= pd.read_csv("../data/features/y.csv")

### Logistic Regression

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
pipe=Pipeline([("Scaler", StandardScaler()),("lr",LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42))])
pipe.fit(X_train, y_train.values.ravel())

Pipeline(steps=[('Scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
                                    random_state=42))])

### Logistic Regression – Default Evaluation (Threshold = 0.5)

In [5]:
y_pred_lr = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

Accuracy: 0.8333333333333334

Confusion Matrix:
 [[385  86]
 [ 14 115]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.82      0.89       471
           1       0.57      0.89      0.70       129

    accuracy                           0.83       600
   macro avg       0.77      0.85      0.79       600
weighted avg       0.88      0.83      0.84       600



### Threshold Tuning for Logistic Regression

In [6]:
# get OOS probabilities
y_prob = pipe.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.3, 0.8, 0.1)

for t in thresholds:
    y_pred_custom = (y_prob >= t).astype(int)
    
    print(f"\n----- Threshold = {t:.1f} ----\n")
    print("Accuracy:", accuracy_score(y_test, y_pred_custom))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_custom))
    print("\nClassification Report:\n", classification_report(y_test, y_pred_custom))


----- Threshold = 0.3 ----

Accuracy: 0.76

Confusion Matrix:
 [[333 138]
 [  6 123]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.71      0.82       471
           1       0.47      0.95      0.63       129

    accuracy                           0.76       600
   macro avg       0.73      0.83      0.73       600
weighted avg       0.87      0.76      0.78       600


----- Threshold = 0.4 ----

Accuracy: 0.8166666666666667

Confusion Matrix:
 [[369 102]
 [  8 121]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.78      0.87       471
           1       0.54      0.94      0.69       129

    accuracy                           0.82       600
   macro avg       0.76      0.86      0.78       600
weighted avg       0.89      0.82      0.83       600


----- Threshold = 0.5 ----

Accuracy: 0.8333333333333334

Confusion Matrix:
 [[385  86]
 [ 14 115]]

Classi

### Final Logistic Regression Model (Selected Threshold = 0.6)

In [7]:
final_threshold = 0.6
y_final_pred = (y_prob >= final_threshold).astype(int)

print("Final Logistic Regression (Threshold = 0.6)")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_final_pred))
print("Classification Report:\n", classification_report(y_test, y_final_pred))

Final Logistic Regression (Threshold = 0.6)
Confusion Matrix:
 [[412  59]
 [ 18 111]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.87      0.91       471
           1       0.65      0.86      0.74       129

    accuracy                           0.87       600
   macro avg       0.81      0.87      0.83       600
weighted avg       0.89      0.87      0.88       600



### Logistic Regression Feature Importance (Coefficients)

In [8]:
lr_coefficients = pipe.named_steps["lr"].coef_[0]

lr_feature_importance = pd.Series(
    lr_coefficients,
    index=X.columns
).sort_values(key=abs, ascending=False)

lr_feature_importance.head(10)

mixing_risk_flag                   1.466702
Product_ID_GastroEase_CR          -0.952621
Product_ID_NeuroCalm_IR            0.941658
granulation_risk_flag              0.633648
Granulation_Moisture_%             0.632641
Granulation_Temp_C_near_high       0.464261
Inlet_Temp_C                      -0.349662
Coating_Spray_Rate                 0.254032
Inlet_Temp_C_near_low              0.222970
Granulation_Moisture_%_near_low    0.174235
dtype: float64

### Alternative Model – Random Forest Classifier

In [9]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train.values.ravel())


RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

### Random Forest Evaluation

In [10]:
y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

Accuracy: 0.985

Confusion Matrix:
 [[471   0]
 [  9 120]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       471
           1       1.00      0.93      0.96       129

    accuracy                           0.98       600
   macro avg       0.99      0.97      0.98       600
weighted avg       0.99      0.98      0.98       600



### Random Forest Feature Importance

In [11]:
rf_feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

rf_feature_importance.head(10)


Product_ID_NeuroCalm_IR         0.181715
Product_ID_GastroEase_CR        0.110605
Granulation_Moisture_%          0.101398
Mixing_Speed_RPM                0.086422
mixing_risk_flag                0.085159
granulation_risk_flag           0.070367
Granulation_Temp_C              0.068133
Inlet_Temp_C                    0.031292
Coating_Spray_Rate              0.030339
Granulation_Temp_C_near_high    0.027675
dtype: float64

### Final Model Selection

Although the Random Forest model achieved very high test performance, the magnitude of improvement compared to Logistic Regression suggests potential overfitting to current data patterns.

Given the regulated pharmaceutical context, the need for interpretability, and stable generalization, **Logistic Regression with a tuned decision threshold of 0.6** was selected as the final model.

Random Forest was retained as a benchmark model to validate the presence of non-linear patterns.


In [13]:
import joblib
import os

# Create a folder for the model if it doesn't exist
os.makedirs("../models", exist_ok=True)

# Save the trained Logistic Regression pipeline
joblib.dump(pipe, "../models/pharma_oos_model.pkl")
print("Model saved successfully!")

Model saved successfully!
